# Publish a Fabric data agent (headless, no pip)

Creates a Fabric **data agent** over the bound lakehouse, selects all tables,
applies QC instructions, sets publish info, and **deploys (publishes)** it via
the AISkill workload API so Foundry IQ can ground on it. Uses only runtime
pre-installed packages (`sempy`, `synapse.ml.fabric`, `notebookutils`) — no pip.
The REST sequence mirrors `fabric-data-agent-sdk` internals exactly.

In [ ]:
import json, time, uuid
import sempy.fabric as fabric
import synapse.ml.fabric.service_discovery as sd
import notebookutils
from sempy.fabric.exceptions import FabricHTTPException

DATA_AGENT_NAME = "{{DATA_AGENT_NAME}}"
LAKEHOUSE_ID = "{{LAKEHOUSE_ID}}"
INSTRUCTIONS = """{{INSTRUCTIONS}}"""

result = {"status": "error", "step": "start"}
try:
    client = fabric.FabricRestClient()
    ws = fabric.get_notebook_workspace_id()
    host = sd.get_fabric_env_config().fabric_env_config.wl_host
    cap = client.get(f"v1/workspaces/{ws}").json().get("capacityId")

    # 1. Create the data agent item (public Fabric API), resolve id by name.
    result["step"] = "create"
    client.post(f"v1/workspaces/{ws}/dataagents", json={"artifactType": "LLMPlugin", "displayName": DATA_AGENT_NAME})
    da_id = None
    for _ in range(20):
        items = client.get(f"v1/workspaces/{ws}/items?type=DataAgent").json().get("value", [])
        match = [i for i in items if i.get("displayName") == DATA_AGENT_NAME]
        if match:
            da_id = match[0]["id"]; break
        time.sleep(3)
    if not da_id:
        raise RuntimeError("data agent item did not appear after create")
    result["dataAgentId"] = da_id

    moniker = str(uuid.uuid4())
    H = {"x-ms-workload-resource-moniker": moniker, "x-ms-ai-assistant-scenario": "data_agent", "x-ms-ai-aiskill-stage": "sandbox"}
    base = f"{host}/webapi/capacities/{cap}/workloads/ML/AISkill/Automatic/workspaces/{ws}/dataagents/{da_id}"

    # 2-3. Get the lakehouse schema and add it as a data source.
    result["step"] = "add_datasource"
    schema_url = (f"{host}/webapi/capacities/{cap}/workloads/ML/AISkill/Automatic/v1/workspaces/{ws}"
                  f"/artifacts/{LAKEHOUSE_ID}/schema?responseSource=live&dataSourceType=lakehouse_tables")
    sch = client.get(schema_url, headers={"x-ms-upstream-artifact-id": da_id, "x-ms-workload-resource-moniker": moniker}).json()
    ds_id = client.post(f"{base}/management/datasources", json=sch["schema"], headers=H).json()["id"]
    result["datasourceId"] = ds_id

    # 4. Select every table.
    result["step"] = "select_all"
    dr = client.get(f"{base}/management/datasources/{ds_id}")
    cfg = dr.json()
    def _sel(elems):
        for e in elems or []:
            if e.get("type") == "lakehouse_tables.table":
                e["is_selected"] = True
            _sel(e.get("children"))
    _sel(cfg.get("elements", []))
    client.put(f"{base}/management/datasources/{ds_id}", json=cfg, headers={**H, "If-Match": dr.headers.get("ETag")})

    # 5. QC instructions (drop dataSources — workload rejects direct edits).
    if INSTRUCTIONS.strip():
        result["step"] = "instructions"
        cr = client.get(f"{base}/management/configuration")
        ccfg = cr.json(); ccfg["additionalInstructions"] = INSTRUCTIONS; ccfg.pop("dataSources", None)
        client.patch(f"{base}/management/configuration", json=ccfg, headers={**H, "If-Match": cr.headers.get("ETag")})

    # 5b. Publish info (Foundry IQ requires it). Mirror the SDK: GET may 404 for a
    #     fresh agent — then create with an empty If-Match.
    result["step"] = "publish_info"
    desc = INSTRUCTIONS.strip()[:200] or "Published by Fabric Demo Gallery"
    try:
        pr = client.get(f"{base}/management/publishing")
        pinfo = pr.json(); pinfo["description"] = desc; petag = pr.headers.get("ETag", "")
    except FabricHTTPException as pe:
        if getattr(pe, "status_code", None) != 404:
            raise
        pinfo = {"description": desc}; petag = ""
    pput = client.put(f"{base}/management/publishing", json=pinfo, headers={"Content-Type": "application/json", "If-Match": petag})
    result["publishingPut"] = pput.status_code

    # 6. Deploy (publish).
    result["step"] = "deploy"
    cr = client.get(f"{base}/management/configuration")
    dep = client.put(f"{base}/management/deploy", headers={"Content-Type": "application/json", "If-Match": cr.headers.get("ETag")})
    result["deployStatus"] = dep.status_code

    result["status"] = "published"
    result["step"] = "done"
except Exception as e:
    import traceback
    result["error"] = f"{type(e).__name__}: {e}"
    result["trace"] = traceback.format_exc()[-1500:]

try:
    with open("/lakehouse/default/Files/publish_result.json", "w") as f:
        json.dump(result, f)
except Exception as fe:
    result["writeError"] = str(fe)

print(json.dumps(result, indent=2))
notebookutils.notebook.exit(json.dumps(result))